# 🧠 Part 2: Kolmogorov-Arnold Network (KAN) Modeling & Multi-Model Benchmarking
### **Project**: KAN-Based Sustainable Fishing Prediction and Decision Support System for Kerala Fisheries

---

## 🎯 Notebook Objectives
1. **Feature Engineering**: Construct lagged CPUE ($t-1, t-2, t-3, t-12$), rolling statistics, climate anomalies, and cyclic trigonometric seasonality.
2. **Zero-Leakage Chronological Data Splitting**:
   - **Training**: 2012 – 2021 (10 years)
   - **Validation**: 2022 – 2023 (2 years)
   - **Out-of-Sample Testing**: 2024 – 2025 (2 years)
3. **Train Baseline Machine Learning Models**: Linear Regression, Ridge, Random Forest, Gradient Boosting, XGBoost, and MLP.
4. **Implement & Train Kolmogorov-Arnold Network (KAN)**: Parameterized with learnable 1D B-spline functions on graph edges.
5. **Multi-Metric Evaluation**: MAE, RMSE, $R^2$, and MAPE.
6. **Explainability & Interpretability Analysis**: Visualizing learned B-spline curves $\phi_{i,j}(x)$ and feature importance rankings.


In [ ]:
import sys
import os
PROJECT_ROOT = os.path.abspath("e:/DA microproject")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
print(f"Project root set: {PROJECT_ROOT}")


In [ ]:
import time
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.fetch_and_clean import generate_fisheries_environmental_dataset
from src.features.engineer_features import engineer_fisheries_features, get_feature_columns
from src.models.kan_model import KolmogorovArnoldNetwork
from src.models.evaluate import train_and_evaluate_all, compute_metrics

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 120
print(f"PyTorch Version: {torch.__version__}, CUDA Available: {torch.cuda.is_available()}")


## 1. Feature Engineering for Target Species (Oil Sardine)
We construct 22 features capturing autoregressive history, environmental forcing, and trophic interactions.

In [ ]:
df_raw = generate_fisheries_environmental_dataset(2012, 2025)
df_feat = engineer_fisheries_features(df_raw, target_species="Oil Sardine")

feature_cols = get_feature_columns()
print(f"Engineered Dataset: {df_feat.shape[0]} records, {len(feature_cols)} input features")
print(f"Features: {feature_cols}")


## 2. Chronological Model Training & Evaluation
We execute strict forward temporal splitting to avoid data leakage.

In [ ]:
eval_results = train_and_evaluate_all(df_feat, epochs=160, lr=0.01)
df_metrics = eval_results["results_df"]

print("--- OUT-OF-SAMPLE TEST PERFORMANCE (2024–2025) ---")
display_cols = ["Model", "Test_MAE", "Test_RMSE", "Test_R2", "Test_MAPE_%", "Train_Time_s"]
print(df_metrics[display_cols].sort_values(by="Test_R2", ascending=False).to_string(index=False))


## 3. Comparative Benchmark Visualizations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. R2 Score Comparison
sns.barplot(data=df_metrics, x='Model', y='Test_R2', ax=axes[0], palette='viridis')
axes[0].set_title("Test R² Score (Higher is Better)", fontsize=12, fontweight='bold')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha='right')
axes[0].set_ylim(0.5, 1.0)

# 2. Test RMSE Comparison
sns.barplot(data=df_metrics, x='Model', y='Test_RMSE', ax=axes[1], palette='magma')
axes[1].set_title("Test RMSE (kg/trip - Lower is Better)", fontsize=12, fontweight='bold')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha='right')

# 3. Test MAPE Comparison
sns.barplot(data=df_metrics, x='Model', y='Test_MAPE_%', ax=axes[2], palette='mako')
axes[2].set_title("Test MAPE (%) (Lower is Better)", fontsize=12, fontweight='bold')
axes[2].set_xticklabels(axes[2].get_xticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.show()


## 4. Test Set Forecast Trajectories: KAN vs Baselines

In [ ]:
test_gt = eval_results["test_ground_truth"]
test_meta = eval_results["test_metadata"].reset_index(drop=True)
kan_preds = df_metrics.loc[df_metrics["Model"] == "Kolmogorov-Arnold Network (KAN)", "Test_Predictions"].values[0]
rf_preds = df_metrics.loc[df_metrics["Model"] == "Random Forest", "Test_Predictions"].values[0]

plt.figure(figsize=(15, 6))
# Filter a specific district (Alappuzha) using positional index for alignment
alappuzha_mask = test_meta['district'] == 'Alappuzha'
alappuzha_pos = test_meta[alappuzha_mask].index

plt.plot(pd.to_datetime(test_meta.loc[alappuzha_pos, 'date']), test_gt[alappuzha_pos], label='Actual CPUE (Ground Truth)', color='black', marker='o', linewidth=2.5)
plt.plot(pd.to_datetime(test_meta.loc[alappuzha_pos, 'date']), kan_preds[alappuzha_pos], label='KAN Prediction', color='crimson', linestyle='--', marker='s', linewidth=2)
plt.plot(pd.to_datetime(test_meta.loc[alappuzha_pos, 'date']), rf_preds[alappuzha_pos], label='Random Forest Prediction', color='forestgreen', linestyle=':', marker='^', linewidth=2)

plt.title("Out-of-Sample CPUE Forecast Comparison (Alappuzha District: 2024–2025)", fontsize=14, fontweight='bold')
plt.xlabel("Date", fontsize=12)
plt.ylabel("CPUE (kg / boat trip)", fontsize=12)
plt.legend(frameon=True, fontsize=11)
plt.show()


## 5. KAN Explainability: Learned 1D B-Spline Activation Curves
A major scientific breakthrough of Kolmogorov-Arnold Networks is their **glass-box interpretability**.
Instead of fixed activation functions on nodes, KAN learns univariate non-linear spline functions $\phi(x)$ on network edges.
Let's inspect the learned response curves for key environmental & operational drivers!

In [ ]:
kan_model = eval_results["trained_models"]["KAN"]
imp = kan_model.get_feature_importance()

df_imp = pd.DataFrame({
    "Feature": feature_cols,
    "Importance_%": imp
}).sort_values(by="Importance_%", ascending=False)

plt.figure(figsize=(12, 6))
sns.barplot(data=df_imp.head(10), x='Importance_%', y='Feature', palette='crest')
plt.title("KAN First-Layer Learned Feature Importance (Spline Weight L1-Norm)", fontsize=14, fontweight='bold')
plt.xlabel("Relative Contribution (%)", fontsize=12)
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

key_features = [
    ("sea_surface_temperature_c", "Sea Surface Temperature (°C)", 0),
    ("fishing_pressure", "Fishing Pressure Index", 8),
    ("upwelling_index", "Coastal Upwelling Index", 5)
]

for ax, (feat_name, feat_title, feat_idx) in zip(axes, key_features):
    x_pts, y_pts = kan_model.evaluate_univariate_splines(feat_idx)
    ax.plot(x_pts, y_pts, color='navy', linewidth=2.5)
    ax.set_title(f"Learned KAN Spline: {feat_title}", fontsize=12, fontweight='bold')
    ax.set_xlabel("Normalized Input Feature", fontsize=11)
    ax.set_ylabel("Learned Activation Response $\phi(x)$", fontsize=11)
    ax.axhline(0, color='gray', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()


### 💡 Key Ecological Insights from KAN Splines
1. **Sea Surface Temperature (SST)**: The learned spline $\phi(\text{SST})$ peaks between normalized values corresponding to $27.8^\circ\text{C} - 28.8^\circ\text{C}$ and shows a steep decline at higher temperatures, capturing the biological thermal threshold of tropical sardines.
2. **Fishing Pressure**: Shows a negative monotonic penalty at elevated effort, reinforcing the non-linear overfishing penalty.
3. **Upwelling**: Displays an exponential-like positive recruitment boost during the monsoon upwelling surge.

We now proceed to **Notebook 03: Decision Support & Optimization System**.